In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from collections import Counter

# Verificar si hay GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando: {device}")

# Cargar datos
df = pd.read_csv('../../data/processed/comments_processed.csv')
df = df.dropna(subset=['text_clean'])

X = df['text_clean'].tolist()
y = df['IsToxic'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

Usando: cpu
Train: 800 | Test: 200


In [2]:
# Construir vocabulario
def build_vocab(texts, max_vocab=5000):
    counter = Counter()
    for text in texts:
        counter.update(text.split())
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, _ in counter.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(X_train)
print(f"Vocabulario: {len(vocab)} palabras")

# Convertir texto a secuencia de índices
def text_to_indices(text, vocab, max_len=100):
    tokens = text.split()[:max_len]
    indices = [vocab.get(t, 1) for t in tokens]  # 1 = <UNK>
    # Padding
    indices += [0] * (max_len - len(indices))
    return indices

MAX_LEN = 100

# Dataset
class CommentDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len):
        self.texts = [text_to_indices(t, vocab, max_len) for t in texts]
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.texts[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float)
        )

train_dataset = CommentDataset(X_train, y_train, vocab, MAX_LEN)
test_dataset = CommentDataset(X_test, y_test, vocab, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

Vocabulario: 3605 palabras
Train batches: 25
Test batches: 7


In [3]:
# Modelo LSTM
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0,
            bidirectional=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)  # *2 por bidireccional

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        output, (hidden, _) = self.lstm(embedded)
        # Concatenar último hidden state de ambas direcciones
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        hidden = self.dropout(hidden)
        return self.fc(hidden).squeeze(1)

# Hiperparámetros
VOCAB_SIZE = len(vocab)
EMBED_DIM = 100
HIDDEN_DIM = 128
N_LAYERS = 2
DROPOUT = 0.3
EPOCHS = 10
LR = 0.001

model = LSTMClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss()

print(model)
print(f"\nParámetros entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

LSTMClassifier(
  (embedding): Embedding(3605, 100, padding_idx=0)
  (lstm): LSTM(100, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)

Parámetros entrenables: 991,541


In [4]:
# Entrenamiento
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0
    for texts, labels in loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        predictions = model(texts)
        loss = criterion(predictions, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # evitar exploding gradients
        optimizer.step()
        total_loss += loss.item()
        correct += ((torch.sigmoid(predictions) >= 0.5) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for texts, labels in loader:
            texts, labels = texts.to(device), labels.to(device)
            predictions = model(texts)
            loss = criterion(predictions, labels)
            total_loss += loss.item()
            preds = (torch.sigmoid(predictions) >= 0.5).long()
            correct += (preds == labels.long()).sum().item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), correct / len(loader.dataset), all_preds, all_labels

# Early stopping
best_val_loss = float('inf')
patience = 3
patience_counter = 0
best_model_state = None

print("Epoch | Train Loss | Train Acc | Val Loss | Val Acc")
print("-" * 55)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, _, _ = evaluate(model, test_loader, criterion)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        patience_counter = 0
    else:
        patience_counter += 1

    print(f"  {epoch+1:2d}  | {train_loss:.4f}     | {train_acc:.4f}    | {val_loss:.4f}   | {val_acc:.4f}")

    if patience_counter >= patience:
        print(f"\nEarly stopping en epoch {epoch+1}")
        break

# Cargar mejor modelo
model.load_state_dict(best_model_state)
print("\nMejor modelo cargado ✅")

Epoch | Train Loss | Train Acc | Val Loss | Val Acc
-------------------------------------------------------
   1  | 0.6906     | 0.5250    | 0.6884   | 0.5400
   2  | 0.6597     | 0.6050    | 0.6678   | 0.6050
   3  | 0.5783     | 0.7113    | 0.6811   | 0.5950
   4  | 0.4859     | 0.7688    | 0.7388   | 0.5950
   5  | 0.3927     | 0.8325    | 0.7044   | 0.6300

Early stopping en epoch 5

Mejor modelo cargado ✅


In [5]:
# Evaluación final
_, _, y_pred_lstm, y_test_lstm = evaluate(model, test_loader, criterion)

print("--- LSTM ---")
print(classification_report(y_test_lstm, y_pred_lstm, target_names=['No tóxico', 'Tóxico']))

train_loss, train_acc, y_pred_train, y_train_lstm = evaluate(model, train_loader, criterion)
train_f1_lstm = f1_score(y_train_lstm, y_pred_train, average='weighted')
test_f1_lstm = f1_score(y_test_lstm, y_pred_lstm, average='weighted')

print(f"\nTrain F1: {train_f1_lstm:.3f}")
print(f"Test F1:  {test_f1_lstm:.3f}")
print(f"Diferencia: {abs(train_f1_lstm - test_f1_lstm):.3f} {'✅ OK' if abs(train_f1_lstm - test_f1_lstm) < 0.05 else '⚠️ Overfitting'}")

--- LSTM ---
              precision    recall  f1-score   support

   No tóxico       0.65      0.69      0.67       108
      Tóxico       0.61      0.55      0.58        92

    accuracy                           0.63       200
   macro avg       0.63      0.62      0.62       200
weighted avg       0.63      0.63      0.63       200


Train F1: 0.921
Test F1:  0.628
Diferencia: 0.293 ⚠️ Overfitting


## LSTM — Resultados

| | LSTM | Ensemble Optuna v2 |
|---|---|---|
| Test F1 | 0.628 | **0.757** |
| Overfitting | 0.293 ⚠️ | **0.035 ✅** |

El LSTM presenta overfitting severo (0.293) con 1000 filas de entrenamiento.
Las redes neuronales recurrentes requieren significativamente más datos para
generalizar bien — típicamente 10k-50k ejemplos mínimo.

Con el dataset actual el ML clásico supera al deep learning en todas las
métricas. Con más datos el LSTM escalaría mejor gracias a su capacidad de
capturar dependencias temporales en el texto.

In [10]:
import joblib
import os

os.makedirs('../../models/model_v1', exist_ok=True)
torch.save(model.state_dict(), '../../models/model_v1/lstm_model.pt')
torch.save(vocab, '../../models/model_v1/lstm_vocab.pt')
print("LSTM guardado ✅")

LSTM guardado ✅


# LSTM en Modelo V3


In [7]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import re

nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\r\n|\r|\n', ' ', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(t) for t in tokens
        if t not in stop_words
        and len(t) > 2
    ]
    return ' '.join(tokens)

# Cargar dataset enriquecido
df = pd.read_csv('../../data/raw/dataset_enriquecido.csv')
df = df.dropna(subset=['Text'])

# El preprocesamiento ya lo tienes, aplícalo
df['text_clean'] = df['Text'].apply(preprocess)

X = df['text_clean'].tolist()
y = df['IsToxic'].astype(int).tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Balanceo train: {pd.Series(y_train).value_counts().to_dict()}")



[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\zulay\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\zulay\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\zulay\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Train: 8331 | Test: 2083
Balanceo train: {0: 4247, 1: 4084}


In [8]:
# Vocabulario con dataset enriquecido
vocab = build_vocab(X_train, max_vocab=10000)  # subimos a 10k por más datos
print(f"Vocabulario: {len(vocab)} palabras")

# Datasets
train_dataset = CommentDataset(X_train, y_train, vocab, MAX_LEN)
test_dataset = CommentDataset(X_test, y_test, vocab, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)  # batch más grande
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

# Reinicializar modelo
model = LSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=100,
    hidden_dim=128,
    n_layers=2,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

# Reentrenar
best_val_loss = float('inf')
patience = 3
patience_counter = 0
best_model_state = None

print("\nEpoch | Train Loss | Train Acc | Val Loss | Val Acc")
print("-" * 55)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, _, _ = evaluate(model, test_loader, criterion)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        patience_counter = 0
    else:
        patience_counter += 1

    print(f"  {epoch+1:2d}  | {train_loss:.4f}     | {train_acc:.4f}    | {val_loss:.4f}   | {val_acc:.4f}")

    if patience_counter >= patience:
        print(f"\nEarly stopping en epoch {epoch+1}")
        break

model.load_state_dict(best_model_state)
print("\nMejor modelo cargado ✅")

Vocabulario: 10000 palabras
Train batches: 131
Test batches: 33

Epoch | Train Loss | Train Acc | Val Loss | Val Acc
-------------------------------------------------------
   1  | 0.5436     | 0.7156    | 0.4242   | 0.8147
   2  | 0.3504     | 0.8458    | 0.3272   | 0.8531
   3  | 0.2778     | 0.8828    | 0.2928   | 0.8867
   4  | 0.2226     | 0.9073    | 0.2650   | 0.8953
   5  | 0.1939     | 0.9240    | 0.2666   | 0.8905
   6  | 0.1593     | 0.9389    | 0.2900   | 0.8925
   7  | 0.1412     | 0.9459    | 0.2792   | 0.8987

Early stopping en epoch 7

Mejor modelo cargado ✅


In [9]:
_, _, y_pred_lstm, y_test_lstm = evaluate(model, test_loader, criterion)

print("--- LSTM con dataset enriquecido ---")
print(classification_report(y_test_lstm, y_pred_lstm, target_names=['No tóxico', 'Tóxico']))

train_loss, train_acc, y_pred_train, y_train_lstm = evaluate(model, train_loader, criterion)
train_f1_lstm = f1_score(y_train_lstm, y_pred_train, average='weighted')
test_f1_lstm = f1_score(y_test_lstm, y_pred_lstm, average='weighted')

print(f"\nTrain F1: {train_f1_lstm:.3f}")
print(f"Test F1:  {test_f1_lstm:.3f}")
print(f"Diferencia: {abs(train_f1_lstm - test_f1_lstm):.3f} {'✅ OK' if abs(train_f1_lstm - test_f1_lstm) < 0.05 else '⚠️ Overfitting'}")

--- LSTM con dataset enriquecido ---
              precision    recall  f1-score   support

   No tóxico       0.88      0.93      0.90      1062
      Tóxico       0.92      0.86      0.89      1021

    accuracy                           0.90      2083
   macro avg       0.90      0.90      0.90      2083
weighted avg       0.90      0.90      0.90      2083


Train F1: 0.974
Test F1:  0.899
Diferencia: 0.076 ⚠️ Overfitting


## LSTM — Impacto del tamaño del dataset

| | 1k filas | 10k filas |
|---|---|---|
| Test F1 | 0.628 | **0.899** |
| Overfitting | 0.293 | 0.076 |
| Val Accuracy | 0.63 | 0.90 |

Con 10x más datos el LSTM superó al ensemble en F1 (0.899 vs 0.757),
confirmando que las redes neuronales recurrentes escalan mejor con
más datos. El overfitting se redujo de 0.293 a 0.076 — aún por encima
del 5% pero significativamente mejor.

Para producción se recomienda el LSTM enriquecido por su mayor F1,
asumiendo que el dataset seguirá creciendo con nuevos comentarios.

In [11]:
os.makedirs('../../models/model_v1', exist_ok=True)
torch.save(model.state_dict(), '../../models/model_v1/lstm_enriched.pt')
torch.save(vocab, '../../models/model_v1/lstm_vocab_enriched.pt')
print("LSTM enriquecido guardado ✅")

LSTM enriquecido guardado ✅
